# 第14回　PyTorch 入門
***
> **前提**: 第13回まで Scikit-learn で機械学習の基礎（Pipeline，正則化，学習曲線など）を学びました。第13回では決定木の `max_depth` による過学習も確認しました。ニューラルネットワークでも epoch 数や層の深さで同様のことが起きます。第15回以降で損失曲線を描いて確認します。
>
> ここから **PyTorch** を使い，深層学習フレームワークの基礎（Tensor，DataLoader，MNIST）を学びます。第15回以降 q14→q15→q16→q17 と PyTorch で一本につながります。

> ⚠️ **この課題で身につけること：コーディングではなく「AI（ディープラーニング）の中身の理解」です。**
>
> コードは AI に書かせても構いません。重要なのは「**なぜその処理を選ぶのか**」「**パラメータや特徴量を変えると結果がどう変わるのか**」を理解し、提出物で示すことです。各問には学習目標を示すタグが付いています。
>
> | タグ | 意味 | あなたがすること |
> |---|---|---|
> | **【骨格】** | 動くコードは与えられている | 設計上の決定点（数値・選択肢・特徴量）だけを変更する |
> | **【選択】** | 適切な手法を選ぶ問題 | 複数候補から選び、**理由**を解答用コードセルに書く |
> | **【実験】** | 試行錯誤の記録 | パラメータ等を変えて結果を表に記録し、**考察**する |
> | **【説明】** | 理解の証跡 | 与えられたコードの各行に `# 説明:` で意味を書く |
>
> コードは原則として完成形ですが、**各問の「核心となる最低限の数行」は `# ★あなたが書く★` として空欄**にしてあります。AI に頼り切らず、要となる処理は自分で書けることも確認します（ボイラープレートは提供済み）。
>
> 各問の **✍️ 解答用コードセル**（`# (1-a)` 形式の変数・文字列）に、設計判断・理由・実験結果・考察を**項目ごとに**記入してください。これが採点対象です。

## 目次
1. Tensor と自動微分
2. MNIST の読み込み
3. MNIST の可視化
4. DataLoader の理解

---

## この回で学ぶこと

### なぜ PyTorch を使うのか

これまで使ってきた scikit-learn は「既成のモデルを使う」ためのライブラリだ。PyTorch は「自分でニューラルネットワークを設計・実装する」ためのフレームワークで，現在の深層学習研究の標準となっている。

| | scikit-learn | PyTorch |
|---|---|---|
| 対象 | 機械学習（線形回帰〜XGBoost） | 深層学習（NN, CNN, Transformer） |
| 柔軟性 | 低（既成モデルを使う） | 高（自由にモデルを設計） |
| 研究での利用 | 特徴量エンジニアリング，ベースライン | 最先端モデルの実装 |

卒業研究で「画像認識」「自然言語処理」「時系列予測」などを扱うなら，PyTorch は必須のスキルだ。

### Tensor とは

Tensor は PyTorch の基本データ構造で，NumPy の `ndarray` に似ているが，以下の点が異なる：
- **GPU で計算できる**（`.to("cuda")`）：大規模モデルの学習を劇的に高速化
- **自動微分（Autograd）** をサポート：ニューラルネットワークの学習に必要な勾配を自動計算

### 自動微分（Autograd）の重要性

ニューラルネットワークの学習には，損失を各パラメータで微分する「誤差逆伝播法（Backpropagation）」が必要だ。これを手計算すると非常に複雑になるが，PyTorch の `requires_grad=True` を使えば自動的に計算できる。

### ミニバッチ学習と DataLoader

全データを一度に学習する「バッチ学習」は：
- メモリに収まらない大規模データでは不可能
- 勾配の計算が遅い

`DataLoader` は大きなデータを小さな「ミニバッチ」に分割して，少しずつ学習できるようにする。`shuffle=True` にすることで，毎エポック異なる順序でデータが供給され，モデルが特定の順序に依存しない学習ができる。

### GPU と CPU の使い分け

```python
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
```

- **GPU**: 深層学習の行列演算を並列処理で高速化（Colab の無料 GPU が使える）
- **CPU**: GPU がない環境でも動作する（今回のような小規模問題では十分）

データとモデルを同じデバイス（GPU か CPU）に置く必要がある。`.to(device)` でデバイスを指定する。

In [ ]:
%pip install -q torch torchvision


In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

DATA_ROOT = "./data"  # MNIST は torchvision が自動ダウンロード


## 問題1　Tensor と自動微分　【説明】
***

### Tensor の基本操作

NumPy の配列に似ているが，型には注意が必要だ：
- `torch.tensor([1, 2, 3])` → デフォルトは int64
- `torch.tensor([1.0, 2.0, 3.0])` → float32（NN で使う型）
- `x.dtype` で型を確認できる

### 自動微分の仕組み

`requires_grad=True` を設定すると，その Tensor を使った計算履歴（計算グラフ）が記録される：

```
x = 5.0 (requires_grad=True)
y = 2x² + 3

y.backward() を呼ぶと：
  dy/dx = 4x を計算 → x=5 なので dy/dx = 20.0
```

これがニューラルネットワークの学習の本質だ：
```
損失L を各重みw で偏微分 → dL/dw を計算（これが勾配）
→ w = w - lr × dL/dw（勾配降下法で重みを更新）
```

PyTorch はこの「勾配の計算」を自動化している。

### 課題

下のコードセルは，Tensor の **生成・形状・変形（reshape / view）・演算** と **自動微分** を確認する **完成形**です。今回は自分でコードを書くのではなく，**Tensor の挙動と自動微分の仕組みを読み解く**のが目的です。

**各行の `# 説明:` の右に，その行が何をしているかを自分の言葉で書いて**ください（コード自体は変更しないこと）。書き終えたらセルを実行し，出力を確認してください。

説明を書くときは，次の問いを意識してください：

- `shape` と `dtype` はそれぞれ何を表すか？ なぜ NN では `float32` を使うのか？
- `reshape` と `view` は何をするか？ 要素数が変わらないのはなぜか？
- `requires_grad=True` を付けると何が記録されるのか？ `backward()` は何を計算するのか？

> **確認**: `y = 2x² + 3` を微分すると `dy/dx = 4x`，x=5 を代入すると 4×5 = 20.0 になるはずです。出力が 20.0 になることを確認してください。

In [ ]:
# 各行の「# 説明:」に自分の言葉で意味を書いてください（コードは変更しない）
# （説明は AI に書かせず、自分で書くこと）

# --- Tensor の生成・形状 ---
t = torch.tensor([1.0, 2.0, 3.0])           # 説明:
print("shape:", t.shape, " dtype:", t.dtype)  # 説明:（shape と dtype は何を表す？）

# --- 形状の変形（reshape / view）---
a = torch.arange(6)                          # 説明:（どんな Tensor ができる？）
b = a.reshape(2, 3)                          # 説明:（reshape は何をする？）
c = b.view(3, 2)                             # 説明:（view と reshape の違いは？）
print("a:", a.shape, " b:", b.shape, " c:", c.shape)

# --- 要素ごとの演算 ---
s = b + 10                                   # 説明:（要素ごとの演算とは？）
print("b + 10 =\n", s)

# --- 自動微分（Autograd）---
x = torch.tensor(5.0, requires_grad=True)    # 説明:（requires_grad=True の意味）
y = 2.0 * x ** 2 + 3.0                        # 説明:
y.backward()                                 # 説明:（backward は何を計算する？）
print("dy/dx =", x.grad.item())              # 説明:（正解は 20.0 になるはず）


In [ ]:
# === ✍️ 問題1 解答（採点対象）===
# 主な提出物は上のコードセルへの # 説明: 記入。以下も記入すること。

# (1-a) `shape` と `dtype` はそれぞれ何を表すか／なぜ NN では `float32` を使うか
answer_1_a = """
"""

# (1-b) `reshape` と `view` は何をするか／要素数が変わらないのはなぜか
answer_1_b = """
"""

# (1-c) `requires_grad=True` を付けると何が記録され、`backward()` は何を計算するか
answer_1_c = """
"""

# (1-d) 確認：`dy/dx` の出力値は？ 理論値 20.0 と一致したか
answer_1_d = """
"""



## 問題2　MNIST データセットの読み込み　【選択+説明】
***

### MNIST とは

MNIST（Modified National Institute of Standards and Technology）は，手書き数字（0〜9）の画像データセットだ：
- 訓練データ：60,000枚
- テストデータ：10,000枚
- 画像サイズ：28×28 ピクセル（グレースケール）
- クラス数：10（数字0〜9）

機械学習の "Hello World" として広く使われており，新しいモデルの動作確認に最適だ。

### `transforms.ToTensor()` が行うこと

`PIL.Image` 形式の画像（ピクセル値 0〜255, uint8）を PyTorch Tensor（値 0.0〜1.0, float32）に変換する：

```
原画像: shape (28, 28), dtype uint8, 値 0〜255
変換後: shape (1, 28, 28), dtype float32, 値 0.0〜1.0

【変換の意味】
・チャネル次元が追加（グレースケールなので 1 チャネル）
・値を 255 で割って正規化（NN の学習を安定化させる）
```

### `batch_size=64` の選び方

バッチサイズはハイパーパラメータの一つ：
- 大きいバッチ（256, 512）：GPU を効率よく使える，勾配が安定，メモリ消費大
- 小さいバッチ（32, 64）：メモリ消費小，ノイズが多く局所最適解から抜けやすいことも
- **64や128が多くの場面でバランスが良い**

### 課題

下のコードセルは，MNIST を読み込んで `DataLoader` を作る **完成形**です。ここでは2つのことをします。

**(1) 選択：`transform` を選ぶ**

`# === ★選択：使う transform を選ぶ★ ===` の `transform_choice` を `"A"`〜`"F"` から選んでください。選んだ値が実際の前処理に反映されます。

- **(A) `ToTensor` のみ** … ピクセル値を 0〜255 から 0.0〜1.0 に変換するだけ
- **(B) `ToTensor` + `Normalize(0.5, 0.5)`** … おおまかに -1〜1 に標準化
- **(C) `ToTensor` + `Normalize`（MNIST の統計 0.1307/0.3081）** … 平均約0・分散約1に標準化
- **(D) `ToTensor` + `RandomRotation(15)`** … ランダムに±15度回転（データ拡張）
- **(E) `ToTensor` + `RandomHorizontalFlip()`** … ランダムに左右反転
- **(F) `ToTensor` + `RandomErasing()`** … 画像の一部をランダムに消す（データ拡張）

> **設計判断**: MNIST の数字認識を学習する目的に対して、最も適切な transform を選び、理由を解答用コードセルに書いてください。**不適切な選択肢も混ざっています**（例えば (E) 左右反転は「6 と 9」「2」などの形を壊すため数字認識には不向きです）。正規化が学習を安定させる理由、データ拡張が有効な場合とそうでない場合にも触れてください。

**(2) 説明＋最低限コーディング：Dataset / DataLoader の構築**

各行の `# 説明:` の右に意味を書いてください（特に `batch_size` と `shuffle` が何をするか）。なお、**DataLoader を作る1行はあなたが書きます**（`# ★あなたが書く★`）。書き終えたら実行し、サンプル数と1件目の情報を確認してください。

In [ ]:
# === ★選択：使う transform を選ぶ（"A"〜"F"）★ ===
transform_choice = "A"

_transforms = {
    "A": transforms.ToTensor(),
    "B": transforms.Compose([transforms.ToTensor(),
                             transforms.Normalize((0.5,), (0.5,))]),
    "C": transforms.Compose([transforms.ToTensor(),
                             transforms.Normalize((0.1307,), (0.3081,))]),
    "D": transforms.Compose([transforms.ToTensor(),
                             transforms.RandomRotation(15)]),
    "E": transforms.Compose([transforms.ToTensor(),
                             transforms.RandomHorizontalFlip()]),
    "F": transforms.Compose([transforms.ToTensor(),
                             transforms.RandomErasing()]),
}
transform = _transforms[transform_choice]

# --- 以下の「# 説明:」に意味を書いてください（batch_size, shuffle に注目）---
train_dataset = datasets.MNIST(
    root=DATA_ROOT, train=True, download=True, transform=transform
)                                                            # 説明:（何を読み込む？ train=True の意味）

# ★あなたが書く★：train_dataset から batch_size=64・shuffle=True の DataLoader を作る（1行）
#   ヒント: DataLoader(データセット, batch_size=..., shuffle=...)
train_loader = ___                                           # 説明:（batch_size と shuffle の意味）

print("サンプル数:", len(train_dataset))                     # 説明:
img, label = train_dataset[0]                                # 説明:（1件取り出すと何が返る？）
print("1件目の画像 shape:", img.shape, " ラベル:", label)    # 説明:


In [ ]:
# === ✍️ 問題2 解答（採点対象）===
# 主な提出物は上のコードセルへの # 説明: 記入。以下も記入すること。

# (2-a) 設計判断：選んだ transform（A〜F）：(　)
# (A) ToTensor のみ
# (B) Normalize(0.5, 0.5)
# (C) RandomRotation
# (D) RandomAffine
# (E) RandomHorizontalFlip
# (F) 複数を組み合わせ
design2_choice = ""

# (2-b) その理由（なぜ正規化が学習を安定させるか／データ拡張が有効・不向きな場合）
answer_2_b = """
"""

# (2-c) 不適切だと思う選択肢を1つ挙げ、なぜ MNIST に不向きか
answer_2_c = """
"""



## 問題3　MNIST の可視化　【骨格】
***

### なぜデータを可視化するのか

「データを見る」ことは機械学習の最も基本的なステップだ。以下を確認することが重要：
1. データが正しく読み込めているか（画像が正しく表示されるか）
2. ラベルと画像が対応しているか
3. データの品質（ぼやけている，傾いている，など）

### `img.squeeze()` の意味

`ToTensor()` 適用後の画像は `(1, 28, 28)`（チャネル×高さ×幅）の shape を持つ。`plt.imshow()` は `(28, 28)` の2次元配列を期待するため，`squeeze()` でチャネル次元（サイズ1の次元）を除去する：

```
(1, 28, 28) → squeeze() → (28, 28)
```

### ピクセル値の確認

`ToTensor()` 適用後のピクセル値は 0.0〜1.0 の範囲になる。これは元の 0〜255 を255で割った値だ。NN への入力は常にこの範囲で行う（大きい値のまま入力すると学習が不安定になる）。

### 課題

下のコードセルは，MNIST の画像を 3×3 のグリッドで表示し，1枚のピクセル値の範囲を出力する **完成済み**のコードです。問題2で `train_dataset` を作ってから、**そのまま実行して出力を観察**してください。

> **観察ポイント**（解答用コードセルに記入）:
> 1. 表示された画像と、その上のラベルは **正しく対応**していますか？（目視で確認）
> 2. ピクセル値の **最小値・最大値** はいくつでしたか？ 問題2で選んだ `transform`（A: ToTensor のみ / B: +Normalize）によって、この範囲はどう変わると考えられますか？

In [ ]:
# === 完成済みコード：そのまま実行して出力を観察してください ===
# （問題2で train_dataset を作成してから実行してください）
fig, axes = plt.subplots(3, 3, figsize=(6, 6))
for ax, idx in zip(axes.flat, range(9)):
    img, label = train_dataset[idx]
    ax.imshow(img.squeeze(), cmap="gray")  # (1,28,28) -> squeeze -> (28,28)
    ax.set_title(f"label: {label}")
    ax.axis("off")
plt.tight_layout()
plt.show()

# 1枚目のピクセル値の範囲を確認
img0, label0 = train_dataset[0]
print("1枚目のラベル:", label0)
print("ピクセル値  min:", img0.min().item(), " max:", img0.max().item())


In [ ]:
# === ✍️ 問題3 解答（採点対象）===

# 観察1（画像とラベルは正しく対応していたか）：
観察1_画像とラベルは正しく対応していたか = """
"""

# 観察2（ピクセル値の min / max は？ 選んだ transform（A / B）でこの範囲はどう変わると考えられるか）：
min = ""
max = ""
transform_による違いの考察 = ""



## 問題4　DataLoader とミニバッチの理解　【骨格+実験】
***

### バッチのデータ構造

DataLoader が返す1バッチのデータ構造を理解することは，第15回以降の実装に直結する：

```
images: shape (64, 1, 28, 28)
  64 = バッチサイズ
   1 = チャネル数（グレースケール）
  28 = 画像の高さ（ピクセル）
  28 = 画像の幅（ピクセル）

labels: shape (64,)
  64個のラベル（0〜9の整数）
```

### Flatten（平坦化）の必要性

全結合層（Fully Connected Layer, `nn.Linear`）は**1次元ベクトル**を入力として受け取る。`(1, 28, 28)` の3次元テンソルをそのまま入れることはできないので，`(784,)` の1次元ベクトルに変形（flatten）する必要がある：

```
28 × 28 = 784 → これが全結合層への入力次元数
```

`view(-1, 784)` の `-1` は「残りの次元を自動計算する」という意味で，バッチサイズを保ったまま flatten できる：
```
(64, 1, 28, 28) → view(-1, 784) → (64, 784)
```

> **第15回との接続**: 第15回では `SimpleMLP` の `forward` メソッドで `x = x.view(-1, 784)` を最初に書く。これがこの flatten 操作だ。

### 課題

下のコードセルは，指定した `batch_size` で `DataLoader` を作り，**1エポックのイテレーション数（バッチの個数）** と **1バッチの shape**、さらに **flatten 前後の shape** を出力する **完成形**です。（問題2で `train_dataset` を作ってから実行してください）

なお、**flatten する核心1行はあなたが書きます**（`# ★あなたが書く★`）。`BATCH_SIZE_EXPERIMENTS` リスト（★印）で **最低5通り**（例：16 / 32 / 64 / 128 / 256）を**1回の実行でループ**します。実行後に表示される `experiment_log_run` の値を，**✍️ 解答用コードセルの `experiment_log`** に転記してください。

> **考察**: `batch_size` を 2倍にすると、1エポックのイテレーション数はどう変わりましたか？ また `images.shape` の最初の数字（バッチサイズ）はどう変わりましたか？ なぜそうなるのか、解答用コードセルに書いてください。
>
> （`flatten 後` の `(N, 784)` の `784 = 28 × 28` は第15回で `nn.Linear` の入力次元になります。）

In [ ]:
# （問題2で train_dataset を作成してから実行してください）
import pandas as pd

# === ★ここを変えて実験する★：比較する batch_size の一覧 ===
BATCH_SIZE_EXPERIMENTS = [16, 32, 64, 128, 256]

experiment_log_run = []
for batch_size in BATCH_SIZE_EXPERIMENTS:
    exp_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    n_iter = len(exp_loader)
    images, labels = next(iter(exp_loader))
    # 実験ループ内の平坦化（第15回の入力次元 784 の確認用）
    flat = images.view(-1, 784)
    row = {
        "batch_size": batch_size,
        "iterations_per_epoch": n_iter,
        "images_shape": str(tuple(images.shape)),
        "flatten_shape": str(tuple(flat.shape)),
    }
    experiment_log_run.append(row)
    print(f"batch_size={batch_size:3d}  iterations={n_iter:4d}  images.shape={images.shape}  flat.shape={flat.shape}")

print("\n--- experiment_log_run（解答欄の experiment_log に転記）---")
print(pd.DataFrame(experiment_log_run))

# --- flatten の核心（1回だけ自分で書く）---
demo_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
images_demo, _ = next(iter(demo_loader))
# ★あなたが書く★：images_demo を (バッチサイズ, 784) に平坦化して flat_demo に代入する（1行）
#   ヒント: images_demo.view(-1, 784)
flat_demo = ___
print("\n[確認] flatten 前:", images_demo.shape, " -> flatten 後:", flat_demo.shape)


In [ ]:
# === ✍️ 問題4 解答（採点対象）===
import pandas as pd


# (4-a) 実験ログ（上のセルで表示した experiment_log_run を転記。5通り以上）
experiment_log = pd.DataFrame([
    {'row': 1, 'batch_size': 16, 'iterations_per_epoch': None, 'images_shape': None, 'flatten_shape': None},
    {'row': 2, 'batch_size': 32, 'iterations_per_epoch': None, 'images_shape': None, 'flatten_shape': None},
    {'row': 3, 'batch_size': 64, 'iterations_per_epoch': None, 'images_shape': None, 'flatten_shape': None},
    {'row': 4, 'batch_size': 128, 'iterations_per_epoch': None, 'images_shape': None, 'flatten_shape': None},
    {'row': 5, 'batch_size': 256, 'iterations_per_epoch': None, 'images_shape': None, 'flatten_shape': None},
])

# (4-b) 考察：batch_size を2倍にするとイテレーション数とバッチサイズはどう変わるか／その理由
answer_4_b = """
"""

